# Этап 7 — Интерпретация модели (SHAP)

**Цель:** объяснить, *почему* модель считает клиента уходящим — превратить «чёрный ящик ~0.69» в понятные факторы риска.

Три задачи:
1. **Глобально** — какие признаки сильнее всего влияют на прогноз оттока (по всем клиентам).
2. **Локально** — разложить риск конкретного клиента по факторам (ядро демо на этапе 8).
3. **Проверить гипотезу** из плана: помогает ли признак `Cluster` предсказанию (метрика с ним против без него).

Финальная модель — LogisticRegression (этапы 5–6). Для линейных моделей SHAP считается точно и быстро (`LinearExplainer`).

In [ ]:
import pandas as pd
import shap
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

## 1. Загружаем финальную модель

Загружаем модель и scaler, сохранённые на этапе 5 (`churn_model.pkl`, `churn_scaler.pkl`) — **не переобучаем**, объясняем именно ту модель, что выбрали. Выборку восстанавливаем тем же детерминированным сплитом (`random_state=101`), scaler применяем через `transform`.

In [ ]:
import joblib

df = pd.read_csv('../data/processed/churn_dataset.csv').set_index('Customer ID')
y = df['churn']
X = df.drop(columns=['churn', 'Cluster_name'])
# .astype(float): get_dummies делает bool-колонки, а SHAP-графики их не переваривают
X = pd.get_dummies(X, columns=['Cluster'], drop_first=True).astype(float)

# та же выборка, что на этапе 5 (тот же random_state)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=101, stratify=y
)

# загружаем финальную модель и её scaler с этапа 5 (НЕ переобучаем)
logreg = joblib.load('../models/churn_model.pkl')
scaler = joblib.load('../models/churn_scaler.pkl')
num_cols = joblib.load('../models/churn_meta.pkl')['num_cols']

# применяем загруженный scaler (transform, не fit)
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols] = scaler.transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print('ROC-AUC:', round(roc_auc_score(y_test, logreg.predict_proba(X_test_scaled)[:, 1]), 3))

## 2. SHAP — глобальная картина

`LinearExplainer` считает вклад каждого признака в прогноз (в пространстве логита). Строим два графика:
- **beeswarm** — каждая точка = клиент, цвет = значение признака, положение по X = вклад в риск оттока (вправо = толкает к «уйдёт»);
- **bar** — средняя по модулю важность признаков.

In [ ]:
explainer = shap.LinearExplainer(logreg, X_train_scaled)
shap_values = explainer(X_test_scaled)

shap.plots.beeswarm(shap_values)

In [ ]:
shap.plots.bar(shap_values)

**Как читать и что видим:**

- **Recency** — главный фактор: чем дольше клиент молчит (красные точки справа), тем выше риск оттока. Ожидаемо и интуитивно верно.
- **Monetary** и **Tenure** — работают в обратную сторону: высокие траты и большой «стаж» **снижают** риск (лояльные, давние клиенты уходят реже).
- Поведенческие признаки (`RecencyRatio`, `PurchaseRate`) и dummy-кластеры вносят мало — подтверждает вывод этапа 6, что почти весь сигнал — в базовых RFM-метриках.

*(SHAP-значения в пространстве логита; значения признаков масштабированы — важно направление вклада, а не абсолютные числа.)*

## 3. SHAP — объяснение конкретного клиента

Для демо (этап 8) нужно уметь ответить «почему ИМЕННО этот клиент в зоне риска». Берём клиента с самым высоким прогнозом оттока и раскладываем его риск по факторам (**waterfall**).

In [ ]:
proba = logreg.predict_proba(X_test_scaled)[:, 1]
i_high = proba.argmax()   # самый рискованный клиент
print('Клиент:', X_test_scaled.index[i_high], '| P(отток) =', round(proba[i_high], 3))
shap.plots.waterfall(shap_values[i_high])

In [ ]:
i_low = proba.argmin()    # самый надёжный клиент
print('Клиент:', X_test_scaled.index[i_low], '| P(отток) =', round(proba[i_low], 3))
shap.plots.waterfall(shap_values[i_low])

**Waterfall** показывает, как из базового прогноза (`E[f(x)]` — средний риск по выборке) складывается риск конкретного клиента: каждый признак толкает вверх (к оттоку) или вниз. Это и есть «топ-факторы риска» для карточки клиента в Streamlit-демо.

## 4. Проверка гипотезы: помогает ли признак `Cluster`?

Из плана проекта: кластер (сегмент из этапа 3) добавлен как признак — предполагалось, что он усилит прогноз. Проверим честно: сравним модель **с** кластером и **без** него на той же тестовой выборке. Для контекста — ещё и голый RFM (3 признака).

In [ ]:
def auc_on(cols):
    m = LogisticRegression(max_iter=1000).fit(X_train_scaled[cols], y_train)
    return round(roc_auc_score(y_test, m.predict_proba(X_test_scaled[cols])[:, 1]), 3)

cluster_cols = [c for c in X.columns if c.startswith('Cluster')]

print('С кластером (все признаки):', auc_on(list(X.columns)))
print('Без кластера             :', auc_on([c for c in X.columns if c not in cluster_cols]))
print('Только RFM (3 признака)  :', auc_on(['Recency', 'Frequency', 'Monetary']))

**Вывод по гипотезе: признак `Cluster` НЕ помогает.**

- с кластером ~0.691, без кластера ~0.692 — разница в пределах шума (даже чуть лучше без него);
- модель на голом RFM (3 признака) даёт **то же** ~0.691.

Причина: кластер сам построен на RFM (этап 3), поэтому **дублирует** информацию, которая уже есть в Recency/Frequency/Monetary. На beeswarm выше это видно — вклад dummy-кластеров маленький. Более того, коэффициенты кластера в логреге нестабильны (у «Лояльных» знак может выйти «к оттоку», хотя реально у них отток низкий) — классический признак **избыточного, коллинеарного** признака.

Это не значит, что кластеризация бесполезна — она ценна **сама по себе** как сегментация для бизнеса (этап 3). Просто как **признак для предсказания оттока** она ничего не добавляет поверх RFM.

## Вывод этапа 7

1. **Что гонит отток** (глобально): главный фактор — `Recency` (давно молчит → риск); `Monetary` и `Tenure` — защитные (ценные и давние клиенты уходят реже). Направления совпадают с бизнес-интуицией — модель не выучила ерунду.
2. **Объяснимость на клиента:** waterfall раскладывает риск конкретного клиента по факторам — готовый механизм для карточки в демо.
3. **Гипотеза про кластер — опровергнута:** как признак он не помогает (дублирует RFM). Ценен как отдельная сегментация, но не как фича churn-модели.

Вместе с этапом 6 это окончательно закрывает вопрос «где сигнал»: почти весь он — в трёх RFM-метриках, и это потолок данных. SHAP сделал модель **прозрачной**, что важнее лишних долей AUC.

Дальше — **этап 8**: Streamlit-демо, где локальные SHAP-объяснения станут топ-факторами риска для каждого клиента.